# Create Databricks Secret Scope & SQL Server Secrets

Creates a Databricks-backed secret scope and stores two secrets for SQL Server authentication.  
Safe to rerun — skips scope creation if it already exists and always updates the secret values.

In [ ]:
# ------------------------------------------------------------
# 1. Configuration — fill in all values before running
# ------------------------------------------------------------

scope_name    = "sql-server-scope"       # name of the secret scope to create
db_user_key   = "sql-server-user"        # secret key name for the database user
db_pass_key   = "sql-server-password"    # secret key name for the database password

db_user_value = "your_database_user"     # actual database username
db_pass_value = "your_database_password" # actual database password

In [ ]:
# ------------------------------------------------------------
# 2. Initialise the Databricks SDK client
#    Authenticates automatically using the notebook context
# ------------------------------------------------------------
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import AclPermission

w = WorkspaceClient()
print(f"Connected to: {w.config.host}")

In [ ]:
# ------------------------------------------------------------
# 3. Create secret scope (skips if it already exists)
# ------------------------------------------------------------
from databricks.sdk.errors import ResourceAlreadyExists

try:
    w.secrets.create_scope(scope=scope_name)
    print(f"Secret scope '{scope_name}' created.")
except ResourceAlreadyExists:
    print(f"Secret scope '{scope_name}' already exists — skipping creation.")

In [ ]:
# ------------------------------------------------------------
# 4. Store secrets (creates or overwrites on every run)
# ------------------------------------------------------------

w.secrets.put_secret(scope=scope_name, key=db_user_key, string_value=db_user_value)
print(f"Secret '{db_user_key}' stored in scope '{scope_name}'.")

w.secrets.put_secret(scope=scope_name, key=db_pass_key, string_value=db_pass_value)
print(f"Secret '{db_pass_key}' stored in scope '{scope_name}'.")

print("\nDone. Reference these secrets in your ingestion notebook as:")
print(f'  dbutils.secrets.get(scope="{scope_name}", key="{db_user_key}")')
print(f'  dbutils.secrets.get(scope="{scope_name}", key="{db_pass_key}")')